In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

# Cambia la ruta al archivo según sea necesario
file_path = r"D:ORDENADOR AFM1\BACTERIAS EXTREMÓFILAS\64.Mediciones Helios desecadas 2024-08-22\64.5.Mediciones Helios-Exp-09-09-2024\AFM Data\64.5-Helios-desecacion-exponencial-09-09-2024_0001_Normal force.fz.cur"

# Leer el archivo y extraer los datos
with open(file_path, 'r') as f:
    lines = f.readlines()

data_start_index = lines.index('[Header end]\n') + 1
data_lines = lines[data_start_index:]

data = []
for line in data_lines:
    if line.strip() and not line.startswith('['):
        data.append(list(map(float, line.strip().split())))

data_array = np.array(data)
delta_data = data_array[:, 0]  # Desplazamiento (nm)
F_data = data_array[:, 1]       # Fuerza (V)

# Función para determinar el tamaño de ventana inicial óptimo
def find_optimal_window_size(F_data, delta_data, min_size=35, max_size=150, step=5):
    slopes = []
    for size in range(min_size, max_size + 1, step):
        x_window = delta_data[:size]
        y_window = F_data[:size]
        
        try:
            params, _ = curve_fit(lambda x, a, b: a * x + b, x_window, y_window)
            slope = abs(params[0])
            slopes.append((size, slope))
        except RuntimeError:
            continue
    
    optimal_window = min(slopes, key=lambda x: x[1])[0]
    return optimal_window

# Determinación automática del tamaño de ventana inicial óptimo
initial_window_size = find_optimal_window_size(F_data, delta_data)
print(f"Initial window size seleccionado automáticamente: {initial_window_size}")

# Cálculo de la media y desviación estándar para la región inicial
mean_force = np.mean(F_data[:initial_window_size])
std_force = np.std(F_data[:initial_window_size])
threshold = std_force * 2

# Identificación del fin de la región de no contacto
for end_index in range(initial_window_size, len(F_data)):
    if np.abs(F_data[end_index] - mean_force) > threshold:
        no_contact_region_end = end_index
        break
else:
    no_contact_region_end = len(F_data)

region_no_contact = delta_data[:no_contact_region_end]
F_no_contact = F_data[:no_contact_region_end]
params, _ = curve_fit(lambda x, a, b: a * x + b, region_no_contact, F_no_contact)
a_fit, b_fit = params

F_base = a_fit * delta_data + b_fit
F_normalized = F_data - F_base
F_normalized -= F_normalized[0]

zero_crossing_indices = np.where(np.diff(np.sign(F_normalized)))[0]

if zero_crossing_indices.size > 0:
    last_zero_crossing_index = zero_crossing_indices[-1] + 1

    delta_contact = delta_data[last_zero_crossing_index:]
    F_contact = F_normalized[last_zero_crossing_index:]

    best_r2 = -1
    best_params = None
    best_segment = None

    min_window_size = 15
    max_window_size = len(F_contact)

    for window_size in range(min_window_size, max_window_size + 1):
        for start in range(len(F_contact) - window_size + 1):
            end = start + window_size
            delta_segment = delta_contact[start:end]
            F_segment = F_contact[start:end]

            try:
                params_segment, _ = curve_fit(lambda x, a, b: a * x + b, delta_segment, F_segment)
                a_fit_segment, b_fit_segment = params_segment
                F_pred = a_fit_segment * delta_segment + b_fit_segment
                r2 = 1 - (np.sum((F_segment - F_pred) ** 2) / np.sum((F_segment - np.mean(F_segment)) ** 2))
                
                if r2 > best_r2:
                    best_r2 = r2
                    best_params = params_segment
                    best_segment = (start, end)
            except Exception:
                continue

    if best_params is not None:
        a_fit_best, b_fit_best = best_params
        F_contact_fit_best = a_fit_best * delta_contact[best_segment[0]:best_segment[1]] + b_fit_best

        plt.figure(figsize=(12, 6))
        plt.subplot(1, 2, 1)
        plt.plot(delta_data, F_data, label='Datos Originales', color='blue')
        plt.axhline(0, color='gray', linestyle='--')
        plt.xlabel('Desplazamiento (nm)')
        plt.ylabel('Fuerza (V)')
        plt.title('Curva Original')
        plt.legend()

        plt.subplot(1, 2, 2)
        plt.plot(delta_data, F_normalized, label='Curva Normalizada', color='orange')
        plt.plot(delta_contact[best_segment[0]:best_segment[1]], F_contact_fit_best, label='Ajuste Lineal Óptimo', color='red')
        plt.axhline(0, color='gray', linestyle='--')
        plt.xlabel('Desplazamiento (nm)')
        plt.ylabel('Fuerza Normalizada (V)')
        plt.title('Curva Normalizada y Ajuste de la Mejor Región de Contacto')
        plt.legend()

        # Añadir anotación de la pendiente en nm/V
        stiffness_V_per_nm = a_fit_best
        stiffness_nm_per_V = 1 / stiffness_V_per_nm if stiffness_V_per_nm != 0 else np.inf
        plt.annotate(f'Pendiente: {stiffness_nm_per_V:.4f} nm/V', 
                     xy=(0.05, 0.95), xycoords='axes fraction', fontsize=10, color='red', ha='left')
        plt.tight_layout()
        plt.show()

        # Cálculo y visualización de la Indentación Total Ajustada
        kcantilver = 38.0
        pendiente_superficie_infinita_dura = 45.0
        F_indentation_nN = F_normalized[last_zero_crossing_index:] * kcantilver * pendiente_superficie_infinita_dura
        indentation_adjusted = delta_data[last_zero_crossing_index:] - (pendiente_superficie_infinita_dura * F_normalized[last_zero_crossing_index:])

        plt.figure(figsize=(12, 6))
        plt.plot(indentation_adjusted, F_indentation_nN, label='Datos de Indentación Ajustada', color='blue')
        plt.axhline(0, color='gray', linestyle='--')
        plt.xlabel('Indentación Ajustada (nm)')
        plt.ylabel('Fuerza de Indentación (nN)')
        plt.title('Análisis de Indentación (Indentación Ajustada)')
        plt.legend()

        # Añadir anotación de la indentación total ajustada
        total_indentacion_nm = indentation_adjusted[-1] - indentation_adjusted[0]
        plt.annotate(f'Indentación Total Ajustada: {total_indentacion_nm:.4f} nm', 
                     xy=(0.05, 0.95), xycoords='axes fraction', fontsize=10, color='blue', ha='left')
        plt.tight_layout()
        plt.show()

    else:
        print("No se encontró un ajuste adecuado en la región de contacto.")
else:
    print("No se encontró un paso por cero en la curva normalizada.")
